# PyPI Dependency Network Analysis

This notebook analyzes the Python package dependency network scraped from PyPI. 
The dataset consists of up to 10,000 nodes, starting from diverse seed packages like `pandas`, `django`, and `fastapi`.

**Goals:**
1. Load the directed graph from the generated `.graphml` file.
2. Identify the most heavily relied-upon packages (In-Degree).
3. Identify packages with the most dependencies (Out-Degree).
4. Calculate the PageRank to find the most "influential" packages in this ecosystem.

In [1]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

# Set pandas display options to make tables look nice in the notebook
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', None)

## 1. Load the Network Data
We are using the `GraphML` format because it natively preserves the directed nature of our graph (Source -> Target).

In [2]:
# Load the graph
file_path = "pypi_multiseed_10k.graphml"
G = nx.read_graphml(file_path)

print("Graph loaded successfully!")
print(f"Number of nodes (packages): {G.number_of_nodes()}")
print(f"Number of edges (dependencies): {G.number_of_edges()}")

Graph loaded successfully!
Number of nodes (packages): 6765
Number of edges (dependencies): 36054


## 2. Define Analysis Functions
Here we define the core functions to calculate degrees and PageRank. Because we are in a notebook, we will return Pandas DataFrames so they render as clean HTML tables.

In [3]:
def get_most_depended_upon(graph, top_n=10):
    """Finds packages that are required by the most other packages (Highest In-Degree)."""
    in_degrees = dict(graph.in_degree())
    sorted_nodes = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)
    return pd.DataFrame(sorted_nodes[:top_n], columns=["Package", "Dependents Count"])

def get_heaviest_dependencies(graph, top_n=10):
    """Finds packages that require the most other packages to function (Highest Out-Degree)."""
    out_degrees = dict(graph.out_degree())
    sorted_nodes = sorted(out_degrees.items(), key=lambda x: x[1], reverse=True)
    return pd.DataFrame(sorted_nodes[:top_n], columns=["Package", "Dependencies Count"])

def calculate_pagerank(graph, top_n=10):
    """Calculates PageRank to find ecosystem importance."""
    pr = nx.pagerank(graph, alpha=0.85)
    sorted_pr = sorted(pr.items(), key=lambda x: x[1], reverse=True)
    return pd.DataFrame(sorted_pr[:top_n], columns=["Package", "PageRank Score"])

## 3. Run Analysis and Display Results

In [4]:
print("Top 10 Most Depended-Upon Packages (Foundational Packages):")
display(get_most_depended_upon(G, 10))

print("\n Top 10 Packages with the Most Dependencies (Heavy Packages):")
display(get_heaviest_dependencies(G, 10))

print("\n Top 10 Most Important Packages (PageRank):")
display(calculate_pagerank(G, 10))

Top 10 Most Depended-Upon Packages (Foundational Packages):


,Package,Dependents Count
0,typing-extensions,1198
1,pytest,1176
2,numpy,731
3,pytest-cov,574
4,sphinx,494
5,requests,414
6,packaging,400
7,pandas,313
8,mypy,311
9,scipy,270



 Top 10 Packages with the Most Dependencies (Heavy Packages):


,Package,Dependencies Count
0,types-boto3,418
1,types-aiobotocore,416
2,types-aioboto3,411
3,aws_cdk.cloudformation-include,223
4,kedro-datasets,197
5,pyobjc,162
6,ag2,144
7,apache-beam,124
8,zenml,111
9,inspect-ai,97



 Top 10 Most Important Packages (PageRank):


,Package,PageRank Score
0,typing-extensions,0.059873
1,pytest,0.023053
2,numpy,0.011513
3,colorama,0.009804
4,pyobjc-core,0.008980
5,sphinx,0.008255
6,packaging,0.007967
7,tomli,0.007309
8,requests,0.006953
9,pytest-cov,0.006214


## 4. Explore Dependency Paths
Let's see how deep the rabbit hole goes. How does one package eventually rely on another?

In [5]:
def find_dependency_path(graph, source_pkg, target_pkg):
    """Finds and prints the shortest dependency chain between two packages."""
    try:
        path = nx.shortest_path(graph, source=source_pkg, target=target_pkg)
        print(f"Path from '{source_pkg}' to '{target_pkg}':\n")
        print(" ➔ ".join(path))
    except nx.NetworkXNoPath:
        print(f"No dependency path exists between '{source_pkg}' and '{target_pkg}'.")
    except nx.NodeNotFound as e:
        print(f"Error: {e}")

# Try changing these variables to packages you know are in your dataset!
source = "django"
target = "numpy"

find_dependency_path(G, source, target)

Path from 'django' to 'numpy':

django ➔ asgiref ➔ pytest ➔ hypothesis ➔ numpy
